# Chapter 8: Code Execution


In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2


## 8.1 E2B Sandbox Setup

Requires `E2B_API_KEY` in `.env`.


In [ ]:
# NOTE: Requires E2B_API_KEY in .env
# Uncomment the following to test with E2B:
# from e2b_code_interpreter import Sandbox
# sandbox = Sandbox()
# print('Sandbox created:', sandbox.sandbox_id)

# For now, inspect the code execution tools
from scratch_agent.tools.code_execution import execute_python, bash_tool, upload_file

print(f"execute_python: {execute_python.description[:80]}")
print(f"bash_tool: {bash_tool.description[:80]}")
print(f"upload_file: {upload_file.description[:80]}")

## 8.2 Code Execution Tools


In [ ]:
import json

# Inspect the tool definitions
for t in [execute_python, bash_tool, upload_file]:
    print(f"--- {t.name} ---")
    print(json.dumps(t.tool_definition, indent=2))
    print()

## 8.3 Agent with Code Execution


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient

# NOTE: Requires E2B_API_KEY in .env for actual execution
# agent = Agent(
#     model=LlmClient(model="gpt-4o-mini"),
#     code_execution="e2b",
#     instructions="You are a helpful coding assistant.",
#     max_steps=5,
# )
# result = await agent.run("Write a function to compute fibonacci(10).")
# print(result.output)

# Without E2B, we can still create an agent and inspect its configuration
agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[],
    instructions="You are a helpful coding assistant.",
    max_steps=5,
)
print(f"Agent name: {agent.name}")
print(f"Tools: {[t.name for t in agent.tools]}")
print("To enable code execution, set code_execution='e2b' and provide E2B_API_KEY.")

## 8.4 Sandbox-Executable Tools


In [ ]:
from scratch_agent.tools.base import tool, FunctionTool

@tool(sandbox_executable=True)
def calculate_stats(numbers: list) -> dict:
    """Calculate basic statistics for a list of numbers."""
    import statistics
    return {
        "mean": statistics.mean(numbers),
        "median": statistics.median(numbers),
        "stdev": statistics.stdev(numbers) if len(numbers) > 1 else 0,
    }

# Inspect the tool
print(f"Name: {calculate_stats.name}")
print(f"Sandbox executable: {calculate_stats.sandbox_executable}")
print(f"Needs context: {calculate_stats.needs_context}")

# Get the source code that would be sent to sandbox
source = calculate_stats.get_source_code()
print(f"\nSource code for sandbox:\n{source}")

## 8.5 Skills System


In [ ]:
from scratch_agent.skills import SkillInfo, discover_skills, generate_skills_prompt

# Inspect the skills API
print(f"SkillInfo fields: {[f for f in SkillInfo.__dataclass_fields__]}")
print(f"\nNote: Skills are discovered from a directory containing Python files with YAML frontmatter.")
print("Example skill file format:")
print("""
---
name: data_analyzer
description: Analyze CSV data and generate summary statistics
---
import pandas as pd

def analyze(file_path: str) -> dict:
    df = pd.read_csv(file_path)
    return df.describe().to_dict()
""")

## 8.6 Agent with Skills


In [ ]:
# Full agent with code execution + skills (requires E2B_API_KEY)
# 
# from scratch_agent.agent import Agent
# from scratch_agent.llm import LlmClient
#
# agent = Agent(
#     model=LlmClient(model="gpt-4o-mini"),
#     code_execution="e2b",
#     skills_path="./skills",
#     instructions="You are an assistant with access to skills and code execution.",
#     max_steps=8,
# )
# result = await agent.run("Use Python to calculate the first 20 Fibonacci numbers.")
# print(result.output)

print("CH08 summary:")
print("- execute_python: Run Python code in E2B sandbox")
print("- bash_tool: Run shell commands in sandbox")
print("- upload_file: Upload files to sandbox")
print("- sandbox_executable: Tools whose source code runs in sandbox")
print("- Skills: YAML-frontmatter Python files discovered and loaded into sandbox")